In [3]:
import pandas as pd

path = r"dataset/play_by_play_2024.csv"
cols = ["posteam", "week", "season_type", "play_type", "epa", "success", "yards_gained"]

pbp = pd.read_csv(path, usecols=cols)
plays= pbp[pbp["play_type"].isin(["run","pass"])].dropna(subset=["epa", "posteam"])

print(len(plays), plays["posteam"].nunique())
print(plays.groupby("season_type")["week"].agg(["min", "max"]))

35050 32
             min  max
season_type          
POST          19   22
REG            1   18


In [6]:
team_week = (
    plays.groupby(["posteam", "week", "season_type"])
    .agg(
        plays = ("epa", "size"),
        yards_per_play=("yards_gained", "mean"),
        success_rate=("success", "mean"),
        epa_per_play=("epa","mean")
    )
    .round(3)
    .reset_index()
)
print(team_week.shape)
print(team_week.head())

(570, 7)
  posteam  week season_type  plays  yards_per_play  success_rate  epa_per_play
0     ARI     1         REG     60           4.550         0.383         0.063
1     ARI     2         REG     62           7.887         0.548         0.347
2     ARI     3         REG     53           5.226         0.415        -0.238
3     ARI     4         REG     58           5.103         0.466        -0.086
4     ARI     5         REG     56           6.482         0.446         0.124


In [9]:
team_type = (
    plays.groupby(["posteam","play_type"])
    .agg(epa_per_play=("epa", "mean"))
    .reset_index()
)

team_style = team_type.pivot(
    index="posteam", columns="play_type", values="epa_per_play"
).reset_index()

team_style.columns = ["posteam", "pass_epa", "run_epa"]
team_style["gap"] = team_style["pass_epa"] - team_style["run_epa"]


print(team_style.shape)
print(team_style.sort_values("gap", ascending=False).head().round(3).reset_index(drop=True))


(32, 4)
  posteam  pass_epa  run_epa    gap
0     MIA     0.062   -0.184  0.245
1     BAL     0.321    0.129  0.193
2     DET     0.247    0.069  0.178
3     CIN     0.143   -0.011  0.155
4      LV    -0.091   -0.237  0.146


In [10]:
team_week.to_csv("team_week.csv", index=False)
team_style.to_csv("team_style.csv", index=False)